## Influência de um programa de fidelidade em um ecommerce

Os dados analisados neste projeto foram obtidos na plataforma [Kaggle - Ecommerce Consumer Behavior Analysis Data](https://www.kaggle.com/datasets/salahuddinahmedshuvo/ecommerce-consumer-behavior-analysis-data/data)

A fim de realizar estudos de análise de dados utilizei esta base de dados de comportamento de compradores de um ecommerce para realizar uma análise de dados exploratória e retirar algum possível insight dos dados.

In [1]:
# Bibliotecas utilizadas
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
ecomm = pd.read_csv('Ecommerce_Consumer_Behavior_Analysis_Data.csv')

# Remover o símbolo de dólar e espaços extras antes de converter para float
ecomm['Purchase_Amount'] = ecomm['Purchase_Amount'].str.replace('$', '').str.strip().astype(float)

# Aqui o pandas substitui os valores nulos por 'Non informed' para colunas do tipo object
for col in ecomm.select_dtypes(include='object').columns:
    ecomm[col] = ecomm[col].fillna('Non informed')

# Aqui o pandas transforma os dados de tempo (que são strings) em datetime, sendo um tipo de dado ideal para trabalhar com datas e horas
ecomm['Time_of_Purchase'] = pd.to_datetime(ecomm['Time_of_Purchase'])

### Entendendo os consumidores - distribuições

In [ ]:
ecomm.describe()

In [18]:
dist_gender = ecomm["Gender"].value_counts()
generos = px.bar(dist_gender,x=list(dist_gender.index),y=dist_gender,template='simple_white',color=dist_gender.index, color_discrete_sequence=px.colors.qualitative.G10_r)
generos.update_layout(xaxis_title="Gênero",
                     yaxis_title="Quantidade de Compras",
                     title="Distribuição de Compras por Gênero",
                     title_x=0.5,
                     title_font=dict(size=20),
                     width=800,
                     height=600,
                     
                     )
generos.show()

In [22]:
# Agrupa os dados por gênero e calcula a soma da receita total
gender_amount = ecomm.groupby('Gender')['Purchase_Amount'].mean().sort_values(ascending=False)
gender = px.bar(gender_amount,x=list(gender_amount.index),y=gender_amount,template='simple_white',color=dist_gender.index, color_discrete_sequence=px.colors.qualitative.G10_r)
gender.update_layout(xaxis_title="Idade",
                     yaxis_title="Receita Total",
                     title="Receita Total por Idade",
                     title_x=0.5,
                     title_font=dict(size=20),
                     width=800,
                     height=600,
                     )
gender.show()

Os dados mostram claramente uma predominância de compradores sendo dos sexos feminino e masculino. Entretanto, o gráfico acima mostra que estes consumidores, em média, gastam menos em suas compras em relação a pessoas de outros sexos.

In [4]:
dist_idade = ecomm["Age"].value_counts()
generos = px.bar(dist_idade,x=list(dist_idade.index),y=dist_idade,template='simple_white')
generos.update_layout(xaxis_title="Idade",
                     yaxis_title="Quantidade de Compras",
                     title="Distribuição de Compras por Idade",
                     title_x=0.5,
                     title_font=dict(size=20),
                     width=800,
                     height=600,
                     
                     )
generos.show()

In [6]:
# Define as faixas etárias e nomes das gerações para categorização 
conditions = [
    (ecomm['Age'] >= 18) & (ecomm['Age'] <= 27),
    (ecomm['Age'] >= 28) & (ecomm['Age'] <= 43),
    (ecomm['Age'] >= 44) & (ecomm['Age'] <= 50)
]

choices = ['Gen Z', 'Gen Y', 'Gen X']

# Cria a nova coluna
ecomm['Generation'] = np.select(conditions, choices, default='Outros')

In [23]:
# Agrupa os dados por geração e calcula a soma da receita total
age_amount = ecomm.groupby('Generation')['Purchase_Amount'].mean().sort_values(ascending=False)
age = px.bar(age_amount,x=list(age_amount.index),y=age_amount,template='simple_white',color=age_amount.index, color_discrete_sequence=px.colors.qualitative.G10_r)
age.update_layout(xaxis_title="Idade",
                     yaxis_title="Receita Média",
                     title="Receita Média por Idade",
                     title_x=0.5,
                     title_font=dict(size=20),
                     )
age.show()

A idade dos compradores está entre 18 e 50 anos, com uma média de 34 anos entre eles. Vemos que há uma distribuição não muito uniforme entre a quantidade de compradores em cada idade, dessa forma foi feita uma segmentação da idade por geração (X, Y e Z) e dessa forma obteve-se a receita média deste por geração, dessa forma, vemos que as gerações gastaram valores médios muito próximos em suas compras.

Em Análise exploratória de dados, um interessante método para se obter informações sobre as colunas é a correlação linear. 

In [25]:
numerical_cols = ecomm.select_dtypes(include='number')
corr = numerical_cols.corr()

fig = px.imshow(corr.round(2), text_auto=True, aspect="auto", color_continuous_scale='RdBu', template='simple_white')

fig.show()

Como vemos acima, a correlação entre os dados numéricos demonstra que a relação linear entre as variáveis é desprezível, indicando que não há uma relação direta entre elas. Dessa forma, fatores mais complexos ou não lineares podem influenciar mais diretamente na decisão de compra do consumidor e/ou os dados não são representativos o suficiente para uma análise detalhada.

Afunilando um pouco a análise e observando a influência do programa de fidelidade, vemos o seguinte contexto

In [26]:

fidelidade_stats = ecomm.groupby('Customer_Loyalty_Program_Member')[['Frequency_of_Purchase', 'Purchase_Amount']].agg(['mean', 'sum'])
fidelidade_stats.reset_index(inplace=True)
fidelidade_stats.columns = ['Fidelidade', 'Frequência Média de Compras', 'Frequência Total de Compras', 'Valor Médio de Compras', 'Valor Total de Compras']
fidelidade_stats['Fidelidade'] = fidelidade_stats['Fidelidade'].replace({True: 'Sim', False: 'Não'})
fidelidade_stats['Frequência Média de Compras'] = fidelidade_stats['Frequência Média de Compras'].round(2)

fidelidade_stats_fig = make_subplots(rows=2, cols=2, subplot_titles=('Frequência Média de Compras', 'Valor Médio de Compras', 'Frequência Total de Compras', 'Valor Total de Compras'), specs=[[{'type':'bar'}, {'type':'bar'}], [{'type':'bar'}, {'type':'bar'}]])
fidelidade_stats_fig.add_trace(go.Bar(x=fidelidade_stats['Fidelidade'], y=fidelidade_stats['Frequência Média de Compras'], name='Frequência Média de Compras'), row=1, col=1)   
fidelidade_stats_fig.add_trace(go.Bar(x=fidelidade_stats['Fidelidade'], y=fidelidade_stats['Valor Médio de Compras'], name='Valor Médio de Compras'), row=1, col=2)
fidelidade_stats_fig.add_trace(go.Bar(x=fidelidade_stats['Fidelidade'], y=fidelidade_stats['Frequência Total de Compras'], name='Frequência Total de Compras'), row=2, col=1)
fidelidade_stats_fig.add_trace(go.Bar(x=fidelidade_stats['Fidelidade'], y=fidelidade_stats['Valor Total de Compras'], name='Valor Total de Compras'), row=2, col=2)

fidelidade_stats_fig.update_layout(title_text='Fidelidade do Cliente', title_x=0.5, title_font=dict(size=20), width=900, height=600,showlegend=False)
fidelidade_stats_fig.update_xaxes(title_text='Fidelidade', row=2, col=1)
fidelidade_stats_fig.update_xaxes(title_text='Fidelidade', row=2, col=2)
fidelidade_stats_fig.update_yaxes(title_text='Freq. Média', row=1, col=1)
fidelidade_stats_fig.update_yaxes(title_text='Valor Médio', row=1, col=2)
fidelidade_stats_fig.update_yaxes(title_text='Freq. Total', row=2, col=1)
fidelidade_stats_fig.update_yaxes(title_text='Valor Total', row=2, col=2)

fidelidade_stats_fig.show()

Vemos pelo gráfico abaixo que o programa de fidelidade não está servindo ao seu propósito, dado que as médias e soma de valor e frequência de compra são menores entre os consumidores que são membros do programa de fidelidade em relação aos que não são. Portanto, um insight importante que se oobtém destes dados é a necessidade de uma revisão dos benefícios oferecidos pelo programa de fidelidade.

In [27]:
# Criando uma figura com subplots para as análises de satisfação
satisfaction_fig = make_subplots(rows=1, cols=2, subplot_titles=["Satisfação (Fidelidade = Sim)", 
                                                                 "Satisfação (Fidelidade = Não)", 
                                                                 ], 
                                 specs=[[{'type': 'box'}, {'type': 'box'}]])

# Adicionando boxplot para Customer_Satisfaction com Customer_Loyalty_Program_Member = True or False
satisfaction_fig.add_trace(
    go.Box(y=ecomm[ecomm['Customer_Loyalty_Program_Member'] == True]['Customer_Satisfaction'], 
           name="Fidelidade = Sim", marker_color='blue'),
    row=1, col=1)

satisfaction_fig.add_trace(
    go.Box(y=ecomm[ecomm['Customer_Loyalty_Program_Member'] == False]['Customer_Satisfaction'], 
           name="Fidelidade = Não", marker_color='red'),
    row=1, col=2)

satisfaction_fig.update_layout(title_text="Análise de Satisfação do Cliente", title_x=0.5, 
                                title_font=dict(size=20), width=800, height=500, showlegend=False)

satisfaction_fig.update_yaxes(title_text="Satisfação", row=1, col=1)
satisfaction_fig.update_yaxes(title_text="Satisfação", row=1, col=2)

satisfaction_fig.show()

Observando a satisfação de compra média entre os clientes que fazem ou não parte do programa de fidelidade, vemos que, apesar dos "Sim" terem um valor um pouco mais alto, a maior concentração de valores ainda é baixa em ambos os casos como pode ser observado pelos boxplots abaixo.

#### Conclusão

Baseado nas análises feitas para entender os efeitos do programa de fidelidade em um ecommerce, observamos que a sua implantação não está sendo lucrativa para os negócios, visto que a satisfação de compra do consumidor é baixa, indicando frustração com o programa de fidelidade utilizado.